# OPRB test-set benchmark — EasyOCR (CRAFT + CRNN)

**Task:** Evaluate the lightweight, 2-stage OCR baseline (**EasyOCR**) on **occluded document images** from OPRB with 100% reproducible settings.

| Component | Choice |
|---|---|
| Dataset | [OPRB — Occluded Pages Restoration Benchmark](https://huggingface.co/datasets/kpurkayastha/OPRB) |
| Allowed input | `test/text/img/` and `test/nontext/img/` only |
| Ground truth | `test/*/gt/` token annotations |
| Model | **EasyOCR (v1.7+)** (CRAFT Text Detector + CRNN Recognizer, ~100M params) |
| Primary metrics | Character Error Rate (CER), Word Error Rate (WER) |
| Additional metrics | Bag-of-words token precision/recall/F1 and Latency (s/page) |
| Reproducibility | Fixed `RANDOM_SEED = 42`, deterministic box sorting, config export |

In [ ]:
%pip -q install "easyocr>=1.7" "huggingface_hub>=0.30" "jiwer>=3.0" "pandas>=2.0" matplotlib Pillow tqdm

In [ ]:
import json, random, re, shutil, tarfile, time, unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from IPython.display import display
from huggingface_hub import hf_hub_download
from jiwer import cer, wer
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Torch:', torch.__version__)

In [ ]:
REPO_ID = "kpurkayastha/OPRB"
MODEL_ID = "EasyOCR (CRAFT+CRNN)"
DATA_ROOT = Path("/content/oprb_test")
OUTPUT_DIR = Path("/content/oprb_easyocr_results")

CATEGORIES = ["text", "nontext"]

# Set to None to evaluate ALL 9,000 pages, or e.g. 5 for smoke test, 300 for validation
MAX_SAMPLES_PER_CATEGORY = 5
RANDOM_SEED = 42
NORMALIZE_CASE = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Categories:', CATEGORIES)
print('Maximum samples per category:', MAX_SAMPLES_PER_CATEGORY)

In [ ]:
def extract_flat(archive_path: str, destination: Path, allowed_suffix: str) -> int:
    destination.mkdir(parents=True, exist_ok=True)
    count = 0
    with tarfile.open(archive_path, mode="r:gz") as archive:
        for member in tqdm(archive, desc=f"Extract {destination.parent.name}/{destination.name}", leave=False):
            if not member.isfile() or not member.name.lower().endswith(allowed_suffix):
                continue
            filename = Path(member.name).name
            target = destination / filename
            if not target.exists():
                stream = archive.extractfile(member)
                if stream is None:
                    continue
                with stream, target.open("wb") as out:
                    shutil.copyfileobj(stream, out)
            count += 1
    return count

def download_test_category(category: str):
    if category not in {"text", "nontext"}:
        raise ValueError(f"Unknown category: {category}")
    for kind, suffix in (("img", ".jpg"), ("gt", ".txt")):
        repo_path = f"test/{category}/{kind}.tar.gz"
        target_dir = DATA_ROOT / category / kind
        existing = list(target_dir.glob(f"*{suffix}")) if target_dir.exists() else []
        expected = 6000 if category == "text" else 3000
        if len(existing) >= expected:
            print(f"Reuse {category}/{kind}: {len(existing)} files")
            continue
        print(f"Downloading {repo_path} ...")
        tar_path = hf_hub_download(repo_id=REPO_ID, repo_type="dataset", filename=repo_path)
        n = extract_flat(tar_path, target_dir, suffix)
        print(f"  {category}/{kind}: {n:,} files -> {target_dir}")

for category in CATEGORIES:
    download_test_category(category)

In [ ]:
def parse_occlusion_metadata(stem: str):
    parts = stem.split("__", 2)
    return {
        "occlusion_type": parts[0] if parts else "unknown",
        "occlusion_density": parts[1] if len(parts) > 1 else "unknown",
    }

def collect_test_pairs(categories):
    pairs = []
    for category in categories:
        image_dir = DATA_ROOT / category / "img"
        gt_dir = DATA_ROOT / category / "gt"
        images = {p.stem: p for p in image_dir.glob("*.jpg")}
        gts = {p.stem: p for p in gt_dir.glob("*.txt")}
        common = sorted(images.keys() & gts.keys())
        print(f"{category}: images={len(images):,}, gt={len(gts):,}, paired={len(common):,}")
        if not common:
            raise RuntimeError(f"No paired files for {category}.")
        if MAX_SAMPLES_PER_CATEGORY is not None:
            rng = random.Random(RANDOM_SEED)
            common = sorted(rng.sample(common, min(MAX_SAMPLES_PER_CATEGORY, len(common))))
        for stem in common:
            pairs.append({
                "category": category,
                "sample_id": stem,
                "image_path": str(images[stem]),
                "gt_path": str(gts[stem]),
                **parse_occlusion_metadata(stem),
            })
    return pairs

pairs = collect_test_pairs(CATEGORIES)
print(f"Selected {len(pairs):,} samples (Seed={RANDOM_SEED}).")

In [ ]:
def read_gt_tokens(gt_path: str):
    tokens = []
    with open(gt_path, "r", encoding="utf-8-sig", errors="replace") as file:
        for line in file:
            fields = line.rstrip("\r\n").split("\t")
            if len(fields) >= 11 and fields[0].strip():
                tokens.append(fields[0].strip())
    return tokens

def clean_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.casefold() if NORMALIZE_CASE else text

def bag_of_words_prf(reference: str, prediction: str):
    ref = Counter(reference.split())
    hyp = Counter(prediction.split())
    matched = sum((ref & hyp).values())
    ref_n, hyp_n = sum(ref.values()), sum(hyp.values())
    precision = matched / hyp_n if hyp_n else (1.0 if ref_n == 0 else 0.0)
    recall = matched / ref_n if ref_n else (1.0 if hyp_n == 0 else 0.0)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1

def compute_metrics(ground_truth: str, prediction: str):
    ref, hyp = clean_text(ground_truth), clean_text(prediction)
    if not ref:
        raise ValueError("Empty GT transcript; cannot compute meaningful CER/WER.")
    p, r, f1 = bag_of_words_prf(ref, hyp)
    return {
        "cer": float(cer(ref, hyp)),
        "wer": float(wer(ref, hyp)),
        "token_precision": p,
        "token_recall": r,
        "token_f1": f1,
        "gt_words": len(ref.split()),
        "pred_words": len(hyp.split()),
    }

## Load EasyOCR Reader and Define Reading-Order Sorter

In [ ]:
import easyocr

device_gpu = torch.cuda.is_available()
reader = easyocr.Reader(['en'], gpu=device_gpu)
print("EasyOCR Reader loaded on", "GPU" if device_gpu else "CPU")

def sort_reading_order(detections, line_threshold=15):
    """
    Sort detections into top-to-bottom, left-to-right reading order.
    Each detection: (bbox, text, conf), where bbox is 4 points: [[x1,y1], [x2,y2], [x3,y3], [x4,y4]].
    """
    if not detections:
        return ""
    
    # Calculate representative (y_top, x_left) for each box
    boxes = []
    for bbox, text, conf in detections:
        if not text.strip():
            continue
        xs = [pt[0] for pt in bbox]
        ys = [pt[1] for pt in bbox]
        boxes.append({
            "y": min(ys),
            "x": min(xs),
            "text": text.strip()
        })
    
    if not boxes:
        return ""
    
    # Sort primarily by y, secondarily by x with line clustering
    boxes.sort(key=lambda b: (round(b['y'] / line_threshold), b['x']))
    return " ".join([b['text'] for b in boxes])

def transcribe_page_easyocr(image_path: str):
    start = time.perf_counter()
    raw_results = reader.readtext(image_path)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed_sec = time.perf_counter() - start
    
    full_transcript = sort_reading_order(raw_results)
    word_count = len(full_transcript.split())
    return full_transcript, elapsed_sec, word_count

In [ ]:
# Smoke test on the first sample
sample = pairs[0]
pred_text, elapsed, words = transcribe_page_easyocr(sample['image_path'])
gt_text = ' '.join(read_gt_tokens(sample['gt_path']))
print('LATENCY:', round(elapsed, 3), 's')
print('PREDICTED WORDS:', words)
print('METRICS:', compute_metrics(gt_text, pred_text))

In [ ]:
RESULTS_CSV = OUTPUT_DIR / 'per_image_results.csv'

if RESULTS_CSV.exists():
    previous = pd.read_csv(RESULTS_CSV).to_dict(orient='records')
    rows = [r for r in previous if r.get('sample_id') in {p['sample_id'] for p in pairs}]
    print(f'Resuming with {len(rows)} previously saved rows.')
else:
    rows = []

finished = {r['sample_id'] for r in rows}
for sample in tqdm(pairs, desc='EasyOCR OPRB evaluation'):
    if sample['sample_id'] in finished:
        continue
    result = dict(sample)
    result['model_id'] = MODEL_ID
    try:
        prediction, elapsed, n_words = transcribe_page_easyocr(sample['image_path'])
        ground_truth = ' '.join(read_gt_tokens(sample['gt_path']))
        result.update({
            'status': 'ok',
            'prediction': prediction,
            'ground_truth': ground_truth,
            'latency_sec': elapsed,
            'generated_tokens': n_words,
            'possibly_truncated': False,
            **compute_metrics(ground_truth, prediction),
        })
    except Exception as exc:
        result.update({'status': 'error', 'error': repr(exc)})
        print('Failure:', sample['sample_id'], repr(exc))
    rows.append(result)
    pd.DataFrame(rows).to_csv(RESULTS_CSV, index=False)

results = pd.DataFrame(rows)
print('Saved:', RESULTS_CSV)
print('Successful:', (results.status == 'ok').sum(), '/', len(results))
display(results[['category', 'occlusion_type', 'cer', 'wer', 'token_f1', 'latency_sec', 'status']].head())

In [ ]:
ok = results.loc[results['status'] == 'ok'].copy()
metric_cols = ['cer', 'wer', 'token_precision', 'token_recall', 'token_f1', 'latency_sec']

overall = {f'mean_{m}': float(ok[m].mean()) for m in metric_cols}
overall.update({'n_success': int(len(ok)), 'n_errors': int(len(results) - len(ok))})

by_category = ok.groupby('category').agg(
    n=('sample_id', 'count'),
    **{f'mean_{c}': (c, 'mean') for c in metric_cols}
).reset_index()

print('OVERALL EasyOCR Metrics:')
print(json.dumps(overall, indent=2))
display(by_category)

by_category.to_csv(OUTPUT_DIR / 'summary_by_category.csv', index=False)

run_config = {
    'dataset': REPO_ID,
    'split': 'test',
    'categories': CATEGORIES,
    'model': MODEL_ID,
    'random_seed': RANDOM_SEED,
    'max_samples_per_category': MAX_SAMPLES_PER_CATEGORY,
    'normalize_case': NORMALIZE_CASE,
    'metrics': overall,
    'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
}
(OUTPUT_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')
shutil.make_archive(str(OUTPUT_DIR), 'zip', root_dir=OUTPUT_DIR)
print('Done! Results zipped at:', f'{OUTPUT_DIR}.zip')